In [ ]:
# ==============================================================================
# STEP 1: SETUP & MERGING (Wave 10 - 2024)
# ==============================================================================
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import os
import warnings

warnings.filterwarnings('ignore')
sns.set_theme(style="whitegrid")

# โหลดและผสานข้อมูล Roster หลักของครัวเรือน
df_sh = pd.read_stata('wave-10-shocks.dta', convert_categoricals=False)
df_dt = pd.read_stata('wave-10-shocks_detail.dta', convert_categoricals=False)
df_24 = pd.merge(df_sh, df_dt, on=['interview__key', 'shocks__id'], how='inner')

print(f"✅ Merged Wave 10 (2024) successfully: {len(df_24)} rows")

# ==============================================================================
# STEP 2: DATA CLEANING & HARMONIZATION
# ==============================================================================
def clean_wave10(df):
    # จัดการ Missing Values มาตรฐาน TVSEP
    df = df.replace([-9, -99, -9.0, -99.0], np.nan)
    
    # คลีนข้อมูลเงิน 3 มิติ (Income loss, Extra Exp, Asset loss)
    money_cols = ['v31105a', 'v31105b', 'v31106a']
    for col in money_cols:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col].astype(str).str.replace('none', '0'), errors='coerce').fillna(0)
            
    # คำนวณมูลค่าความเสียหายรวมตามโจทย์
    df['impact_3way'] = df['v31105a'] + df['v31105b'] + df['v31106a']
    df['impact_2way'] = df['v31105a'] + df['v31106a']
    
    # จัดกลุ่ม Shock Type
    shock_mapping = {
        10: 'agricultural', 11: 'agricultural', 63: 'agricultural', 55: 'agricultural',
        1: 'demographic', 2: 'demographic', 3: 'demographic', 24: 'demographic',
        5: 'economics', 6: 'economics', 18: 'economics', 21: 'economics', 22: 'economics', 62: 'economics',
        8: 'social', 70: 'social', 77: 'economics'
    }
    df['shocks_Group'] = df['shocks__id'].map(shock_mapping).fillna('others')
    df['survey_year'] = 2024
    
    # ปรับชื่อคอลัมน์ Coping Strategy เพื่อทำความเข้าใจร่วมกันในแผงข้อมูล
    coping_map = {
        'v31108a__1': 'coping_savings', 
        'v31108a__2': 'coping_insurance', 
        'v31108a__3': 'coping_informal_borrow', 
        'v31108a__4': 'coping_formal_borrow', 
        'v31108a__5': 'coping_assets', 
        'v31108a__7': 'coping_gov_help'
    }
    df = df.rename(columns=coping_map)
    return df

df_24 = clean_wave10(df_24)

# ==============================================================================
# STEP 3: MASTER 8 GRAPHS DESCRIPTIVE STATISTICS
# ==============================================================================
def run_8_graphs_wave10(df, year):
    output_dir = f"graphs_wave_{year}"
    if not os.path.exists(output_dir): os.makedirs(output_dir)

    # จัดการ Labels สำหรับ Recovery และ Consumption
    def map_rec(m):
        if pd.isna(m): return np.nan
        if m < 12: return "less than 1 year"
        if m == 12: return "1 year"
        if 12 < m < 90: return "more than 1 year, but recovered"
        return "not yet recovered" if m >= 90 else np.nan
    
    df['recovery_std'] = df['v31112a'].apply(map_rec)
    df['cons_label'] = df['v31111'].map({1: "Yes (Reduced)", 2: "No (Did not reduce)"})
    coping_vars = [c for c in df.columns if c.startswith('coping_')]

    plots = [
        ('G1_Frequency', lambda: sns.countplot(data=df, x='shocks_Group', palette='viridis')),
        ('G2_Impact_3way', lambda: sns.barplot(data=df, x='shocks_Group', y='impact_3way', estimator=np.mean, palette='magma')),
        ('G3_Loss_2way', lambda: sns.barplot(data=df, x='shocks_Group', y='impact_2way', estimator=np.mean, palette='flare')),
        ('G4_CopingUsage', lambda: df[coping_vars].sum().sort_values().plot(kind='barh', color='skyblue')),
        ('G5_CopingIntensity', lambda: sns.countplot(data=df, x=df[coping_vars].sum(axis=1), palette='plasma')),
        ('G6_RecoveryMonths', lambda: sns.histplot(data=df[df['v31112a'] < 90], x='v31112a', bins=20, kde=True, color='teal')),
        ('G7_RecoveryStd', lambda: sns.countplot(data=df, x='recovery_std', order=["less than 1 year", "1 year", "more than 1 year, but recovered", "not yet recovered"], palette='Set2')),
        ('G8_Consumption', lambda: sns.countplot(data=df[df['cons_label'].notnull()], x='cons_label', palette='Set1'))
    ]

    for name, func in plots:
        plt.figure(figsize=(10, 5))
        func()
        plt.title(f"{name.replace('_',' ')} ({year})")
        if any(x in name for x in ['G1','G2','G3']): plt.xticks(rotation=45)
        plt.savefig(f"{output_dir}/{name}_{year}.png", dpi=300, bbox_inches='tight')
        plt.show()
        
    print(f"✅ All 8 graphs for {year} saved in folder: {output_dir}")

run_8_graphs_wave10(df_24, 2024)

# ส่งออกข้อมูลคลีนของปี 2024 เพื่อนำไปทำ Master Panel ต่อไป
df_24.to_csv('shocks_2024_cleaned.csv', index=False)